# 06-07. Carbon Isotopes in PSLNMAD  
# PSLNMAD で \( \delta^{13}\mathrm{C} \) と \( \Delta^{14}\mathrm{C} \) を考える

**Ocean Box Modeling with Python / 海洋ボックスモデル入門**

## 今日の目的 / Goals

06-06 では、PSLNMAD に DIC, \( \mathrm{PO4} \), O2 を導入しました。  
In 06-06, we introduced DIC, \( \mathrm{PO4} \), and O2 into PSLNMAD.

この Notebook では、炭素同位体を導入します。  
In this notebook, we introduce carbon isotopes.

扱う同位体は次です。  
We treat:

```text
δ13C
Δ14C
```

```text
δ13C
Δ14C
```

この 2 つは、炭素循環を見るための非常に重要なトレーサーです。  
These two tracers are very important for understanding the carbon cycle.

> **\( \delta^{13}\mathrm{C} \) は主に生物ポンプと水塊混合を、\( \Delta^{14}\mathrm{C} \) は主にベンチレーション・隔離時間を考える手がかりになる。**  
> **\( \delta^{13}\mathrm{C} \) helps us think about the biological pump and water-mass mixing, while \( \Delta^{14}\mathrm{C} \) helps us think about ventilation and isolation time.**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (8, 4.8)

## 1. Question: なぜ同位体を入れるのか / Why add isotopes?

DIC, \( \mathrm{PO4} \), O2 だけでも、炭素循環の基本は見えます。  
DIC, \( \mathrm{PO4} \), and O2 already show the basic carbon cycle.

しかし、それだけでは次の問いに答えにくいです。  
However, they are not enough to answer:

```text
その炭素は生物ポンプで増えたのか？
その水はどれくらい古いのか？
A と D は同じ深層水なのか？
D の炭素はどれくらい大気から隔離されていたのか？
```

```text
Did the carbon increase because of the biological pump?
How old is the water?
Are A and D the same kind of deep water?
How long was carbon in D isolated from the atmosphere?
```

そこで \( \delta^{13}\mathrm{C} \) と \( \Delta^{14}\mathrm{C} \) を使います。  
This is why we use \( \delta^{13}\mathrm{C} \) and \( \Delta^{14}\mathrm{C} \).

## 2. \( \delta^{13}\mathrm{C} \) の直感 / Intuition for \( \delta^{13}\mathrm{C} \)

植物プランクトンは、相対的に軽い \( ^{12}\mathrm{C} \) を取り込みやすいです。  
Phytoplankton preferentially take up the lighter isotope \( ^{12}\mathrm{C} \).

そのため、表層に残る DIC は相対的に \( ^{13}\mathrm{C} \) に富み、\( \delta^{13}\mathrm{C} \) が高くなりやすいです。  
Therefore, remaining surface DIC becomes relatively enriched in \( ^{13}\mathrm{C} \), and \( \delta^{13}\mathrm{C} \) tends to be high.

一方、内部では有機物が再無機化され、低い \( \delta^{13}\mathrm{C} \) の炭素が戻ります。  
In the interior, remineralization returns carbon with low \( \delta^{13}\mathrm{C} \).

```text
surface biological uptake:
    δ13C tends to increase

interior remineralization:
    δ13C tends to decrease
```

## 3. \( \Delta^{14}\mathrm{C} \) の直感 / Intuition for \( \Delta^{14}\mathrm{C} \)

\( ^{14}\mathrm{C} \) は放射壊変します。  
\( ^{14}\mathrm{C} \) decays radioactively.

そのため、表層から長く隔離された水ほど \( \Delta^{14}\mathrm{C} \) は低くなります。  
Therefore, water isolated from the surface for a long time tends to have lower \( \Delta^{14}\mathrm{C} \).

```text
young water:
    high Δ14C

old water:
    low Δ14C
```

PSLNMAD では、A と D の違いを見る上で \( \Delta^{14}\mathrm{C} \) が重要です。  
In PSLNMAD, \( \Delta^{14}\mathrm{C} \) is important for distinguishing A and D.

## 4. PSLNMAD の準備 / Prepare PSLNMAD

In [ ]:
boxes = ["P", "S", "L", "N", "M", "A", "D"]
surface_boxes = ["P", "S", "L", "N"]
interior_boxes = ["M", "A", "D"]

VOCN_TOTAL = 1.292e18
VOLUME = {
    "P": 0.18 * VOCN_TOTAL,
    "S": 0.05 * VOCN_TOTAL,
    "L": 0.12 * VOCN_TOTAL,
    "N": 0.03 * VOCN_TOTAL,
    "M": 0.27 * VOCN_TOTAL,
    "A": 0.12 * VOCN_TOTAL,
}
VOLUME["D"] = VOCN_TOTAL - sum(VOLUME.values())

volumes = np.array([VOLUME[b] for b in boxes])
idx = {b: i for i, b in enumerate(boxes)}

Q_DEFAULT = 2.0e7
DT = 8.64e4
SEC_PER_YEAR = 365 * 86400
DT_YEAR = DT / SEC_PER_YEAR

pathway = [("N", "A"), ("A", "D"), ("D", "S"), ("S", "P"), ("P", "L"), ("L", "N")]
exchanges_default = [("S", "M", 0.3e7), ("L", "M", 0.2e7)]

pd.DataFrame({
    "Box": boxes,
    "Layer": ["surface", "surface", "surface", "surface", "mid-depth", "Atlantic deep pathway", "deep"],
    "Volume fraction": [VOLUME[b] / VOCN_TOTAL for b in boxes],
})

## 5. 輸送行列 / Transport matrix

In [ ]:
def build_flux_matrix(pathway, Q, boxes):
    F = np.zeros((len(boxes), len(boxes)))
    local_idx = {b: i for i, b in enumerate(boxes)}
    for source, target in pathway:
        i = local_idx[target]
        j = local_idx[source]
        F[i, j] += Q
        F[j, j] -= Q
    return F

def build_flux_matrix_with_exchange(pathway, Q, boxes, exchanges=None):
    F = build_flux_matrix(pathway, Q, boxes)
    local_idx = {b: i for i, b in enumerate(boxes)}
    if exchanges is None:
        exchanges = []
    for a, b, q in exchanges:
        ia = local_idx[a]
        ib = local_idx[b]
        F[ib, ia] += q
        F[ia, ia] -= q
        F[ia, ib] += q
        F[ib, ib] -= q
    return F

F = build_flux_matrix_with_exchange(pathway, Q_DEFAULT, boxes, exchanges_default)
pd.DataFrame(F, index=boxes, columns=boxes)

## 6. Ideal Age から \( \Delta^{14}\mathrm{C} \) proxy を作る / Build a \( \Delta^{14}\mathrm{C} \) proxy from Ideal Age

まず、06-05 と同じように Ideal Age を計算します。  
First, we calculate Ideal Age as in 06-05.

その後、簡単な式で \( \Delta^{14}\mathrm{C} \) proxy を作ります。  
Then we make a simple \( \Delta^{14}\mathrm{C} \) proxy.

```text
older age → lower Δ14C
```

ここでは教育用に、

```text
Δ14C = -1000 * (1 - exp(-age / 8267))
```

とします。  
For teaching, we use:

```text
Δ14C = -1000 * (1 - exp(-age / 8267))
```

In [ ]:
def one_step_transport(c, F):
    return c + (F @ c) * DT / volumes

def one_step_age(age, F):
    new = one_step_transport(age, F)

    for b in interior_boxes:
        new[idx[b]] += DT_YEAR

    for b in surface_boxes:
        new[idx[b]] = 0.0

    return new

def run_ideal_age(years=3000, Q=Q_DEFAULT, exchanges=exchanges_default):
    F_local = build_flux_matrix_with_exchange(pathway, Q, boxes, exchanges)
    age = np.zeros(len(boxes))
    hist = []

    for day in range(years * 365 + 1):
        if day % 365 == 0:
            row = {"year": day / 365}
            for i, b in enumerate(boxes):
                row[b] = age[i]
            hist.append(row)
        age = one_step_age(age, F_local)

    return pd.DataFrame(hist)

age_df = run_ideal_age(years=3000)
final_age = age_df.iloc[-1]

D14C_proxy = {}
for b in boxes:
    age = final_age[b]
    D14C_proxy[b] = -1000.0 * (1.0 - np.exp(-age / 8267.0))

pd.DataFrame({
    "Box": boxes,
    "Ideal age [yr]": [final_age[b] for b in boxes],
    "Delta14C proxy [permil]": [D14C_proxy[b] for b in boxes],
})

In [ ]:
plt.figure()
for b in ["M", "A", "D"]:
    plt.plot(age_df["year"], age_df[b], label=b)
plt.xlabel("Time [yr]")
plt.ylabel("Ideal age [yr]")
plt.title("Ideal age: M, A, D")
plt.legend()
plt.grid(True)
plt.show()

plt.figure()
plt.scatter([final_age[b] for b in boxes], [D14C_proxy[b] for b in boxes])
for b in boxes:
    plt.text(final_age[b], D14C_proxy[b], b)
plt.xlabel("Ideal age [yr]")
plt.ylabel("Delta14C proxy [permil]")
plt.title("Older water has lower Delta14C")
plt.grid(True)
plt.show()

### Mini exercise 1 / ミニ演習 1

Ideal Age が大きい box では、\( \Delta^{14}\mathrm{C} \) proxy はどうなりましたか。  
What happens to the \( \Delta^{14}\mathrm{C} \) proxy in boxes with larger Ideal Age?

A と D の違いを説明してください。  
Explain the difference between A and D.

## 7. \( \delta^{13}\mathrm{C} \) proxy を作る / Build a \( \delta^{13}\mathrm{C} \) proxy

次に、簡単な \( \delta^{13}\mathrm{C} \) proxy を作ります。  
Next, we build a simple \( \delta^{13}\mathrm{C} \) proxy.

考え方は次です。  
The idea is:

```text
surface biological uptake increases δ13C
interior remineralization decreases δ13C
older/interior water tends to have lower δ13C
```

ここでは、06-06 で作った DIC-O2 の直感に合わせて、

```text
δ13C = 2.0 - 0.0007 * age
```

のような単純な式を使います。  
Here, consistent with the DIC-O2 intuition in 06-06, we use a simple formula:

```text
δ13C = 2.0 - 0.0007 * age
```

In [ ]:
d13C_proxy = {}
for b in boxes:
    age = final_age[b]
    d13C_proxy[b] = 2.0 - 0.0007 * age

iso_df = pd.DataFrame({
    "Box": boxes,
    "Ideal age [yr]": [final_age[b] for b in boxes],
    "delta13C proxy [permil]": [d13C_proxy[b] for b in boxes],
    "Delta14C proxy [permil]": [D14C_proxy[b] for b in boxes],
})
iso_df

In [ ]:
plt.figure()
plt.scatter(iso_df["Ideal age [yr]"], iso_df["delta13C proxy [permil]"])
for _, row in iso_df.iterrows():
    plt.text(row["Ideal age [yr]"], row["delta13C proxy [permil]"], row["Box"])
plt.xlabel("Ideal age [yr]")
plt.ylabel("delta13C proxy [permil]")
plt.title("Simple delta13C proxy")
plt.grid(True)
plt.show()

## 8. A と D の同位体差 / Isotope differences between A and D

A と D を分ける理由は、同位体でも重要です。  
Separating A and D is also important for isotope interpretation.

A は N から入る比較的新しい深層経路です。  
A is the relatively young deep pathway from N.

D はより古く、表層から長く隔離されやすい深層水です。  
D is older deep water, more isolated from the surface.

したがって、一般には、

```text
D has lower Δ14C than A
D may have lower δ13C than A
```

となりやすいです。  
Therefore, in general:

```text
D has lower Δ14C than A
D may have lower δ13C than A
```

In [ ]:
ad_df = iso_df[iso_df["Box"].isin(["A", "D"])].copy()
ad_df

In [ ]:
plt.figure()
plt.bar(ad_df["Box"], ad_df["Delta14C proxy [permil]"])
plt.ylabel("Delta14C proxy [permil]")
plt.title("Delta14C proxy: A vs D")
plt.grid(True, axis="y", alpha=0.3)
plt.show()

plt.figure()
plt.bar(ad_df["Box"], ad_df["delta13C proxy [permil]"])
plt.ylabel("delta13C proxy [permil]")
plt.title("delta13C proxy: A vs D")
plt.grid(True, axis="y", alpha=0.3)
plt.show()

### Mini exercise 2 / ミニ演習 2

A と D で、\( \Delta^{14}\mathrm{C} \) はどちらが低くなりましたか。  
Which is lower in \( \Delta^{14}\mathrm{C} \), A or D?

これはベンチレーション年齢の違いと整合的ですか。  
Is this consistent with the difference in ventilation age?

## 9. ベンチレーション強度 Q の感度 / Sensitivity to ventilation strength Q

ベンチレーションが強くなると、内部水は若くなり、\( \Delta^{14}\mathrm{C} \) は高くなりやすいです。  
When ventilation becomes stronger, interior water becomes younger and \( \Delta^{14}\mathrm{C} \) tends to be higher.

ここでは Q を変えます。  
Here we vary Q.

In [ ]:
Q_list = [0.5e7, 1.0e7, 2.0e7, 4.0e7]
summary_Q = []

for q in Q_list:
    h = run_ideal_age(years=3000, Q=q, exchanges=exchanges_default)
    last = h.iloc[-1]
    A_age = last["A"]
    D_age = last["D"]
    summary_Q.append({
        "Q": q,
        "Age A": A_age,
        "Age D": D_age,
        "Delta14C A": -1000.0 * (1.0 - np.exp(-A_age / 8267.0)),
        "Delta14C D": -1000.0 * (1.0 - np.exp(-D_age / 8267.0)),
        "delta13C A": 2.0 - 0.0007 * A_age,
        "delta13C D": 2.0 - 0.0007 * D_age,
    })

q_df = pd.DataFrame(summary_Q)
q_df

In [ ]:
plt.figure()
plt.plot(q_df["Q"], q_df["Delta14C A"], "o-", label="A")
plt.plot(q_df["Q"], q_df["Delta14C D"], "o-", label="D")
plt.xlabel("Q [m3/s]")
plt.ylabel("Delta14C proxy [permil]")
plt.title("Ventilation strength and Delta14C")
plt.legend()
plt.grid(True)
plt.show()

plt.figure()
plt.plot(q_df["Q"], q_df["Age A"], "o-", label="A")
plt.plot(q_df["Q"], q_df["Age D"], "o-", label="D")
plt.xlabel("Q [m3/s]")
plt.ylabel("Ideal age [yr]")
plt.title("Ventilation strength and Ideal Age")
plt.legend()
plt.grid(True)
plt.show()

### Mini exercise 3 / ミニ演習 3

Q が大きくなると、A と D の \( \Delta^{14}\mathrm{C} \) はどう変化しましたか。  
How do \( \Delta^{14}\mathrm{C} \) values in A and D change when Q increases?

これは「ベンチレーションが強い」という直感と整合的ですか。  
Is this consistent with the intuition of stronger ventilation?

## 10. 生物ポンプと \( \delta^{13}\mathrm{C} \) / Biological pump and \( \delta^{13}\mathrm{C} \)

\( \delta^{13}\mathrm{C} \) は水塊年齢だけでなく、生物ポンプにも影響されます。  
\( \delta^{13}\mathrm{C} \) is affected not only by water age but also by the biological pump.

ここでは、教育用に「再無機化が強いほど \( \delta^{13}\mathrm{C} \) が低くなる」という proxy を考えます。  
Here, for teaching, we use a proxy where stronger remineralization lowers \( \delta^{13}\mathrm{C} \).

06-06 と同じく、再無機化は DIC を増やし、O2 を減らします。  
As in 06-06, remineralization increases DIC and decreases O2.

したがって、低 O2 の水では \( \delta^{13}\mathrm{C} \) も低くなりやすいと考えます。  
Therefore, low-O2 water may also tend to have lower \( \delta^{13}\mathrm{C} \).

In [ ]:
# simple combined proxy using age and a rough O2-like term
O2_proxy = {b: 260.0 - 0.04 * final_age[b] for b in boxes}
d13C_bio_proxy = {b: 2.0 - 0.0005 * final_age[b] + 0.003 * (O2_proxy[b] - 260.0) for b in boxes}

bio_iso_df = pd.DataFrame({
    "Box": boxes,
    "Age": [final_age[b] for b in boxes],
    "O2 proxy": [O2_proxy[b] for b in boxes],
    "delta13C bio proxy": [d13C_bio_proxy[b] for b in boxes],
})
bio_iso_df

In [ ]:
plt.figure()
plt.scatter(bio_iso_df["O2 proxy"], bio_iso_df["delta13C bio proxy"])
for _, row in bio_iso_df.iterrows():
    plt.text(row["O2 proxy"], row["delta13C bio proxy"], row["Box"])
plt.xlabel("O2 proxy")
plt.ylabel("delta13C bio proxy [permil]")
plt.title("Low O2 tends to accompany low delta13C")
plt.grid(True)
plt.show()

## 11. 2 つの同位体を合わせて読む / Reading two isotopes together

\( \delta^{13}\mathrm{C} \) と \( \Delta^{14}\mathrm{C} \) は、同じものを見ているわけではありません。  
\( \delta^{13}\mathrm{C} \) and \( \Delta^{14}\mathrm{C} \) do not show exactly the same thing.

```text
δ13C:
    biological pump
    remineralization
    water-mass mixing

Δ14C:
    ventilation
    isolation time
    radioactive decay
```

```text
δ13C:
    生物ポンプ
    再無機化
    水塊混合

Δ14C:
    ベンチレーション
    隔離時間
    放射壊変
```

2 つを一緒に見ることで、深層炭素貯蔵の原因をよりよく考えられます。  
By reading both together, we can better infer the causes of deep carbon storage.

In [ ]:
plt.figure()
plt.scatter(iso_df["Delta14C proxy [permil]"], iso_df["delta13C proxy [permil]"])
for _, row in iso_df.iterrows():
    plt.text(row["Delta14C proxy [permil]"], row["delta13C proxy [permil]"], row["Box"])
plt.xlabel("Delta14C proxy [permil]")
plt.ylabel("delta13C proxy [permil]")
plt.title("Two-isotope space")
plt.grid(True)
plt.show()

### Mini exercise 4 / ミニ演習 4

A と D は、2 isotope space で同じ場所にありますか。  
Are A and D located at the same position in two-isotope space?

違いがある場合、それは何を意味しますか。  
If they differ, what does it mean?

## 12. 06-07 のまとめ / Summary of 06-07

この Notebook で学んだことは次です。  
What we learned:

1. \( \delta^{13}\mathrm{C} \) は、生物ポンプ・再無機化・水塊混合の手がかりになる。  
   \( \delta^{13}\mathrm{C} \) provides clues about the biological pump, remineralization, and water-mass mixing.

2. \( \Delta^{14}\mathrm{C} \) は、ベンチレーション・隔離時間・水塊年齢の手がかりになる。  
   \( \Delta^{14}\mathrm{C} \) provides clues about ventilation, isolation time, and water-mass age.

3. A は D より若く、\( \Delta^{14}\mathrm{C} \) が高くなりやすい。  
   A is younger than D and tends to have higher \( \Delta^{14}\mathrm{C} \).

4. D はより古く、\( \Delta^{14}\mathrm{C} \) が低くなりやすい。  
   D is older and tends to have lower \( \Delta^{14}\mathrm{C} \).

5. 2 つの同位体を組み合わせると、ベンチレーションと生物ポンプを区別しやすくなる。  
   Combining two isotopes helps distinguish ventilation effects from biological pump effects.

## Key message

> **PSLNMAD では、A と D の同位体差を通じて、北大西洋起源深層水と古い深層水の違いを診断できる。**  
> **In PSLNMAD, isotope differences between A and D help diagnose the difference between North Atlantic-origin deep water and older deep water.**

## 13. 課題 / Exercises

### 課題 1 / Exercise 1

\( \delta^{13}\mathrm{C} \) は何を考える手がかりになるか。  
What does \( \delta^{13}\mathrm{C} \) help us understand?

### 課題 2 / Exercise 2

\( \Delta^{14}\mathrm{C} \) は何を考える手がかりになるか。  
What does \( \Delta^{14}\mathrm{C} \) help us understand?

### 課題 3 / Exercise 3

A と D で \( \Delta^{14}\mathrm{C} \) が異なる理由を説明せよ。  
Explain why \( \Delta^{14}\mathrm{C} \) differs between A and D.

### 課題 4 / Exercise 4

ベンチレーションが強くなると \( \Delta^{14}\mathrm{C} \) はどう変わるか。  
How does \( \Delta^{14}\mathrm{C} \) change when ventilation becomes stronger?

### 課題 5 / Exercise 5

\( \delta^{13}\mathrm{C} \) と \( \Delta^{14}\mathrm{C} \) を同時に見る意味を説明せよ。  
Explain why it is useful to examine \( \delta^{13}\mathrm{C} \) and \( \Delta^{14}\mathrm{C} \) together.

## 14. 次へ / Next step

次の Notebook では、PSLNMAD の総まとめを行います。  
In the next notebook, we summarize the PSLNMAD model.

```text
06-08 PSLNMAD Summary and Integrated Exercises
```

輸送、トレーサー、Ideal Age、炭素循環、同位体をつなげて、7-box モデル全体を復習します。  
We connect transport, tracers, Ideal Age, carbon cycling, and isotopes to review the whole seven-box model.